# 4k Video Upscaler Colab (Real-ESRGAN)

Adapted from: [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN)

Made with ❤️ by: [yuvraj108c](https://github.com/yuvraj108c)

Github repository: https://github.com/yuvraj108c/4k-video-upscaler-colab

In [1]:
# ==============================================================================
#@markdown BƯỚC 1: KIỂM TRA TOÀN BỘ CẤU HÌNH PHẦN CỨNG HỆ THỐNG
# ==============================================================================
print("="*40)
print("🔍 1. KIỂM TRA THÔNG TIN PHẦN CỨNG HỆ THỐNG")
print("="*40)

# Kiểm tra phiên bản Python hiện tại
import sys
print(f"📦 Phiên bản Python của Colab: {sys.version.split()[0]}")

# Kiểm tra thông tin CPU
print("\n🖥️  Thông tin CPU:")
!lscpu | grep -E "Model name|Core\(s\) per socket|Socket\(s\)|CPU\(s\):"

# Kiểm tra bộ nhớ RAM hệ thống
print("\n💾 Thông tin bộ nhớ RAM hệ thống:")
with open('/proc/meminfo', 'r') as f:
    lines = f.readlines()
for line in lines:
    if 'MemTotal' in line or 'MemFree' in line or 'MemAvailable' in line:
        print(f"   {line.strip()}")

# Kiểm tra card đồ họa GPU và bộ nhớ VRAM
print("\n🎮 Thông tin Card đồ họa GPU (Khuyên dùng cho Real-ESRGAN):")
import torch
gpu_available = torch.cuda.is_available()
if gpu_available:
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"   [OK] Có GPU hỗ trợ: {gpu_name}")
    print(f"   📥 Tổng bộ nhớ VRAM: {total_vram:.2f} GB")
    # Hiện bảng thông số chi tiết của NVIDIA
    print("\n   📊 Bảng trạng thái NVIDIA-SMI chi tiết:")
    !nvidia-smi --query-gpu=name,memory.total,memory.free,driver_version --format=csv
else:
    print("   ❌ Không tìm thấy GPU CUDA! Hệ thống đang chạy ở chế độ [Chỉ CPU] (CPU-only).")
    print("   ⚠️  CẢNH BÁO: Chạy Real-ESRGAN bằng CPU sẽ RẤT CHẬM!")


🔍 1. KIỂM TRA THÔNG TIN PHẦN CỨNG HỆ THỐNG
📦 Phiên bản Python của Colab: 3.13.16

🖥️  Thông tin CPU:
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Core(s) per socket:                      1
Socket(s):                               1
NUMA node0 CPU(s):                       0,1

💾 Thông tin bộ nhớ RAM hệ thống:
   MemTotal:       13286944 kB
   MemFree:         9343984 kB
   MemAvailable:   12380280 kB

🎮 Thông tin Card đồ họa GPU (Khuyên dùng cho Real-ESRGAN):
   ❌ Không tìm thấy GPU CUDA! Hệ thống đang chạy ở chế độ [Chỉ CPU] (CPU-only).
   ⚠️  CẢNH BÁO: Chạy Real-ESRGAN bằng CPU sẽ RẤT CHẬM!


# 1. Setup (~1 minute)

In [2]:

# 1. Tải repo phiên bản cấu trúc độc lập (Python 3.13) của bạn
!git clone "https://github.com/datphan-cpu/Real-ESRGAN_py3.13"
%cd "Real-ESRGAN_py3.13"

# 2. Cài đặt các thư viện lõi từ requirements của bạn (Bỏ hoàn toàn gfpgan, facexlib, basicsr bên ngoài)
!pip install -q -r requirements.txt

# 3. Cài đặt thư viện xử lý video/hình ảnh bổ trợ tương thích Python 3.13
!pip install -q ffmpeg-python opencv-python Pillow tqdm "numpy<2"

# 4. Ép Python nhận diện các package cục bộ (basicsr, gfpgan, facexlib) trong repo
import sys
import os
sys.path.append(os.getcwd())

# 5. Tạo thư mục output chuẩn tuyệt đối ngoài màn hình chính để lưu ảnh/video
os.makedirs("/content/output", exist_ok=True)


Cloning into 'Real-ESRGAN_py3.13'...
remote: Enumerating objects: 897, done.
remote: Counting objects: 100% (897/897), done.
remote: Compressing objects: 100% (398/398), done.
remote: Total 897 (delta 448), reused 886 (delta 437), pack-reused 0 (from 0)
Receiving objects: 100% (897/897), 5.49 MiB | 11.41 MiB/s, done.
Resolving deltas: 100% (448/448), done.
/content/Real-ESRGAN_py3.13
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 78.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 17.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.2 r


# 2. Mount drive (optional)

In [3]:
from google.colab import drive
mount_drive=False #@param{type:"boolean"}

if mount_drive:
  drive.mount('/content/gdrive/')

# 3. Upscale video

- The upscaled video will be saved to `output_dir`
- If google drive is mounted, it will be also saved at `MyDrive/Upscaled Videos (REAL-ESRGAN)`


In [7]:
input_type = "Image" #@param ["Image", "Video"]
video_path = "/content/AIC_meme_OR.png" #@param {type:"string"}
output_dir = "/content/output/" #@param {type:"string"}
resolution = "4 x original" #@param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)", "2 x original", "3 x original", "4 x original"] {type:"string"}
model = "RealESRGAN_x4plus" #@param ["RealESRGAN_x4plus", "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3"]

import os
import cv2
import subprocess

assert os.path.exists(video_path), f"File does not exist: {video_path}"
os.makedirs(output_dir, exist_ok=True)

# 1. Retrieve dimensions dynamically (Image vs Video)
if input_type == "Image":
  img = cv2.imread(video_path, cv2.IMREAD_UNCHANGED)
  assert img is not None, f"Could not read image: {video_path}"
  media_height, media_width = img.shape[:2]
else:
  video_capture = cv2.VideoCapture(video_path)
  media_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
  media_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
  video_capture.release()
  assert media_width > 0 and media_height > 0, f"Could not read video stream: {video_path}"

video_width = media_width
video_height = media_height
final_width = None
final_height = None
aspect_ratio = float(video_width / video_height)

# 2. Calculate output resolutions
match resolution:
  case "FHD (1920 x 1080)":
    final_width = 1920
    final_height = 1080
  case "2k (2560 x 1440)":
    final_width = 2560
    final_height = 1440
  case "4k (3840 x 2160)":
    final_width = 3840
    final_height = 2160
  case "2 x original":
    final_width = 2 * video_width
    final_height = 2 * video_height
  case "3 x original":
    final_width = 3 * video_width
    final_height = 3 * video_height
  case "4 x original":
    final_width = 4 * video_width
    final_height = 4 * video_height

if aspect_ratio == 1.0 and "original" not in resolution:
  final_height = final_width

if aspect_ratio < 1.0 and "original" not in resolution:
  temp = final_width
  final_width = final_height
  final_height = temp

scale_factor = max(final_width / video_width, final_height / video_height)
isEven = int(video_width * scale_factor) % 2 == 0 and int(video_height * scale_factor) % 2 == 0

# scale_factor needs to be even
while isEven == False:
  scale_factor += 0.01
  isEven = int(video_width * scale_factor) % 2 == 0 and int(video_height * scale_factor) % 2 == 0

scale_factor = round(scale_factor, 2)
print(f"Upscaling [{input_type}] from {video_width}x{video_height} to {final_width}x{final_height}, scale_factor={scale_factor}")

file_name_with_ext = os.path.basename(video_path)
base_name, in_ext = os.path.splitext(file_name_with_ext)

# 3. Branching execution by input_type
if input_type == "Image":
  # Image Pipeline: native .png output maintaining maximum quality
  !python inference_realesrgan.py -n {model} -i '{video_path}' -o '{output_dir}' --outscale {scale_factor} --face_enhance --ext png

  raw_output_path = os.path.join(output_dir, f"{base_name}_out.png")
  final_file_name = f"{base_name}_upscaled_{final_width}_{final_height}.png"
  final_file_path = os.path.join(output_dir, final_file_name)

  if os.path.exists(raw_output_path):
    subprocess.run(f"cp '{raw_output_path}' '{final_file_path}'", shell=True)
  else:
    raw_output_path = os.path.join(output_dir, f"{base_name}_out{in_ext}")
    final_file_path = raw_output_path
    final_file_name = os.path.basename(final_file_path)

  print(f"Upscaled image saved to: {final_file_path}")

else:
  # Video Pipeline: inference_realesrgan_video.py with ffmpeg cropping
  !python inference_realesrgan_video.py -n {model} -i '{video_path}' -o '{output_dir}' --outscale {scale_factor}

  upscaled_video_path = os.path.join(output_dir, f"{base_name}_out.mp4")
  final_file_name = f"{base_name}_upscaled_{final_width}_{final_height}.mp4"
  final_file_path = os.path.join(output_dir, final_file_name)

  # Crop to fit if custom resolution
  if "original" not in resolution:
    print("Cropping video to fit...")
    command = f"ffmpeg -loglevel error -hwaccel cuda -y -i '{upscaled_video_path}' -c:v h264_nvenc -filter:v 'crop={final_width}:{final_height}:(in_w-{final_width})/2:(in_h-{final_height})/2' -c:v libx264 -pix_fmt yuv420p '{final_file_path}'"
    subprocess.run(command, shell=True)
  else:
    command = f"cp '{upscaled_video_path}' '{final_file_path}'"
    subprocess.run(command, shell=True)

  if os.path.exists(upscaled_video_path):
    subprocess.run(f"rm '{upscaled_video_path}'", shell=True)

  print(f"Upscaled video saved to: {final_file_path}")

# 4. Google Drive Sync
if 'mount_drive' in globals() and mount_drive:
  drive_folder = "MyDrive/Upscaled Images (REAL-ESRGAN)" if input_type == "Image" else "MyDrive/Upscaled Videos (REAL-ESRGAN)"
  save_directory_drive = f"/content/gdrive/{drive_folder}"
  os.makedirs(save_directory_drive, exist_ok=True)

  command = f"cp '{final_file_path}' '{save_directory_drive}/{final_file_name}'"
  subprocess.run(command, shell=True)
  print(f"Saved to drive: /{drive_folder}/{final_file_name}")


Upscaling [Image] from 523x382 to 2092x1528, scale_factor=4.0
Downloading: "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth" to /content/Real-ESRGAN_py3.13/weights/RealESRGAN_x4plus.pth

100% 63.9M/63.9M [00:00<00:00, 310MB/s]
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Testing 0 AIC_meme_OR
Upscaled image saved to: /content/output/AIC_meme_OR_upscaled_2092_1528.png


# 4. Disconnect runtime

In [ ]:
from google.colab import runtime

disconnect_when_finish = False  #@param{type:"boolean"}

if disconnect_when_finish:
  runtime.unassign()